In [1]:
from mpi4py import MPI
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dolfinx import fem as fe
import os
from itertools import product
from tqdm import tqdm
import time
import seaborn as sns
 
from data_io import save_pickle, load_pickle
from metrics import calculate_analysis_metrics
from dca_utils import*
 
from fourd_var import run_assimilation, setup_data_assimilation, run_data_assimilation
from plotting import plot_mixed_function, plot_comparison1, plot_comparison2

sns.set_palette("bright")
plt.style.use("mystyle1.mplstyle")


In [3]:
def run_parameter_crossval(pickle_path, problem_params, prob, solver_params, 
                       obs_space_freq, obs_time_freq, final_time,
                       obs_std_values, inflation_factor_values, station_ids, 
                       output_dir='da_output', verbose=True, var_type='dci_wme'):
    """
    Run parameter cross validation over observation standard deviation and inflation factor values.
    """
    
    def run_single_experiment(obs_std, inflation_factor):
        """Run a single experiment and return results."""
        exp_name = f"obs_std_{obs_std}_inflation_{inflation_factor}"
        
        try:
            # Setup and run assimilation
            result = setup_data_assimilation(
                pickle_path=pickle_path,
                problem_params=problem_params.copy(),
                prob=prob,
                obs_std=obs_std,
                obs_space_freq=obs_space_freq,
                obs_time_freq=obs_time_freq,
                station_ids=station_ids,
                final_time=final_time,
                inflation_factor=inflation_factor,
                print_setup=True
            )
            save_pickle("setup_result.pkl", result)
            
            analysis, run_bathy = run_assimilation(
                result['problem_params'], solver_params, result['stations'],
                result['y_obs'], result['obs_per_window'], result['obs_time_indices'],
                result['H'], result['covs'], result['hb'], 'sloped_beach',
                cost_function_type=var_type
            )
            
        
            var_rmse, var_misfit = calculate_analysis_metrics(
                analysis_name=var_type,
                analysis_data=analysis,
                save_first=True,
            )

            # Save results
            output_filename = f'{var_type}_analysis_{exp_name}.pkl'
            save_pickle(output_filename, analysis)
            
            return {
                'obs_std': obs_std,
                'inflation_factor': inflation_factor,
                'analysis': analysis,
                'setup_result': result,
                'output_file': output_filename
            }
            
        except SystemExit as e:
            return {
                'obs_std': obs_std,
                'inflation_factor': inflation_factor,
                'error': f'Solver convergence failure (SystemExit: {e.code})',
                'error_type': 'convergence_failure'
            }
        except Exception as e:
            return {
                'obs_std': obs_std,
                'inflation_factor': inflation_factor,
                'error': str(e),
                'error_type': 'general_exception'
            }
    
    def print_progress(exp_count, total, exp_name, start_time, result):
        """Print experiment progress."""
        elapsed = time.time() - start_time
        if 'error' in result:
            print(f"  ✗ Failed after {elapsed:.2f}s: {result.get('error', 'Unknown error')}")
        else:
            print(f"  ✓ Completed in {elapsed:.2f}s - Saved to {result['output_file']}")
    
    def print_summary(all_results):
        """Print final summary statistics."""
        total = len(all_results)
        successful = sum(1 for r in all_results.values() if 'error' not in r)
        convergence_failures = sum(1 for r in all_results.values() 
                                 if r.get('error_type') == 'convergence_failure')
        other_failures = total - successful - convergence_failures
        
        print(f"\nParameter cross validation completed!")
        print(f"Successful: {successful}/{total}")
        if convergence_failures > 0:
            print(f"Convergence failures: {convergence_failures}")
        if other_failures > 0:
            print(f"Other failures: {other_failures}")
        
        if convergence_failures > 0:
            print(f"\nConvergence failure parameters:")
            for result in all_results.values():
                if result.get('error_type') == 'convergence_failure':
                    print(f"  obs_std={result['obs_std']}, inflation_factor={result['inflation_factor']}")
    
    # Main execution
    os.makedirs(output_dir, exist_ok=True)
    all_results = {}
    param_combinations = list(product(obs_std_values, inflation_factor_values))
    
    if verbose:
        print(f"Starting {len(param_combinations)} experiments")
        print(f"obs_std: {obs_std_values}")
        print(f"inflation_factor: {inflation_factor_values}")
        print("=" * 80)
    
    # Run all experiments
    for i, (obs_std, inflation_factor) in enumerate(param_combinations, 1):
        exp_name = f"obs_std_{obs_std}_inflation_{inflation_factor}"
        
        if verbose:
            print(f"\nExperiment {i}/{len(param_combinations)}: {exp_name}")
            start_time = time.time()
        
        result = run_single_experiment(obs_std, inflation_factor)
        all_results[exp_name] = result
        
        if verbose:
            print_progress(i, len(param_combinations), exp_name, start_time, result)
    
    # Save summary and print results
    save_pickle('parameter_crossval_summary.pkl', all_results)
    
    if verbose:
        print("=" * 80)
        print_summary(all_results)
    
    return all_results

In [7]:
if __name__ == "__main__":
    # Configuration
    CONFIG = {
        'obs_std_values': [0.01, 0.05, 0.1, 0.5, 1.0, 1.5],
        'inflation_factor_values': [0.1, 0.5, 1.5, 4.0],
        'final_time': Time.THREE_DAYS.seconds,
        'window_size': Time.ONE_HOUR.seconds,
        'dt': 600,  # 10 minutes
        'run_true': True,
    }
    
    # Problem parameters
    problem_params = {
        'dt': CONFIG['dt'],
        't': 0,
        't_final': CONFIG['final_time'],
        'num_steps': int(np.ceil(CONFIG['final_time'] / CONFIG['dt'])),
        'num_windows': CONFIG['final_time'] // CONFIG['window_size'],
        'fric_law': 'linear',
        'alpha': 2.0 * np.pi / Time.TWELVE_HOURS.seconds,
        'sol_var': 'h'
    }
    
    # Solver parameters
    solver_params = {
        "rtol": 1e-5,
        "atol": 1e-6, 
        "max_it": 10,
        "relaxation_parameter": 1.0,
        "ksp_type": "gmres",
        "pc_type": "ilu",
        "ksp_ErrorIfNotConverged": False
    }
    
    # Station configuration
    # station_ids = {
    #     'method': 'region',
    #     'params': {
    #         'bounds': {'x': (1000, 6000.0), 'y': (1000, 6000)},
    #         'criteria': 'center'
    #     }
    # }
    # stat_ids = [21, 64, 81, 103, 136]
    stat_ids = np.arange(1, 20, 2).tolist() + np.arange(103,129,2).tolist()
    stat_ids.sort() 
    station_ids = {
        'method': 'indices', 
        'params': stat_ids
    }
    # Setup problem and generate true signal
    prob, solver = create_problem_solver(problem_params, "sloped_beach", true_signal=True, verbose=False)
    
    if CONFIG['run_true']:
        assert problem_params['num_steps'] == int(np.ceil(problem_params['t_final'] / problem_params['dt']))
        true_solver = get_true_signal(solver, 'sloped_beach', solver_params, 1)

    
    # Run parameter cross validation
    results = run_parameter_crossval(
        pickle_path='true_signal.pkl',
        problem_params=problem_params,
        prob=prob,
        solver_params=solver_params,
        obs_space_freq=2,  # Legacy parameter, will be removed
        obs_time_freq=1,
        final_time=CONFIG['final_time'],
        obs_std_values=CONFIG['obs_std_values'],
        inflation_factor_values=CONFIG['inflation_factor_values'],
        station_ids=station_ids,
        output_dir='da_output',
        verbose=True,
        var_type='dci_wme'
    )
    
    # Print summary
    def print_results_summary(results):
        """Print a clean summary of experiment results."""
        print("\nParameter sweep results summary:")
        print("-" * 40)
        
        for exp_name, output in results.items():
            if 'error' not in output:
                print(f"✓ {exp_name}: SUCCESS")
            elif output.get('error_type') == 'convergence_failure':
                print(f"✗ {exp_name}: CONVERGENCE FAILURE")
            else:
                print(f"✗ {exp_name}: ERROR - {output['error'][:50]}...")
        
        # Convergence failure recommendations
        convergence_failures = sum(1 for r in results.values() 
                                 if r.get('error_type') == 'convergence_failure')
        
        if convergence_failures > 0:
            print(f"\n⚠️  {convergence_failures} experiments failed due to convergence issues.")
            print("Consider adjusting:")
            print("• obs_std and inflation_factor values")
            print("• Solver tolerances or max iterations")
            print("• Observation setup parameters")
    
    print_results_summary(results)

Starting 24 experiments
obs_std: [0.01, 0.05, 0.1, 0.5, 1.0, 1.5]
inflation_factor: [0.1, 0.5, 1.5, 4.0]

Experiment 1/24: obs_std_0.01_inflation_0.1
Selected 23 stations using method 'indices'
Station cells: [  1   3   5   7   9  11  13  15  17  19 103 105 107 109 111 113 115 117
 119 121 123 125 127]
DATA ASSIMILATION EXPERIMENT SETUP

Input Parameters:
  Pickle path: true_signal.pkl
  Observation std deviation: 0.01
  Observation time frequency: 1
  Final time: 259200
  Inflation factor: 0.1

Problem Parameters:
  dt: 600
  t: 0
  t_final: 259200
  num_steps: 6
  num_windows: 72
  fric_law: mannings
  alpha: 0.0001454441043328608
  sol_var: h

Calculated Dimensions:
  State dimension: 1296
  Observation dimension: 23
  Total time steps: 433
  Observations per window: 6
  Number of observation obs_stations: 23

Matrix Information:
  Observation matrix H shape: (23, 1296)
  Background covariance B shape: (1296, 1296)
  Observation covariance R shape: (23, 23)
  Predicted covariance L 

Processing windows:  69%|██████▉   | 50/72 [02:45<01:41,  4.62s/window]

Optimization failed:


Optimization completed:
  Success: False
  Status: 2
  Message: ABNORMAL: 
  Final cost: 8.409605e+04
  Iterations: 1
  Function evaluations: 36
  Gradient norm at solution: 3.521548e+00

------------------------------------------------------------



Processing windows: 100%|██████████| 72/72 [04:13<00:00,  3.52s/window]


DCI_WME RMSE: 0.2257161808, DCI_WME, Relative Misfit: 0.0951
  ✓ Completed in 253.93s - Saved to dci_wme_analysis_obs_std_0.01_inflation_0.1.pkl

Experiment 2/24: obs_std_0.01_inflation_0.5
Selected 23 stations using method 'indices'
Station cells: [  1   3   5   7   9  11  13  15  17  19 103 105 107 109 111 113 115 117
 119 121 123 125 127]
DATA ASSIMILATION EXPERIMENT SETUP

Input Parameters:
  Pickle path: true_signal.pkl
  Observation std deviation: 0.01
  Observation time frequency: 1
  Final time: 259200
  Inflation factor: 0.5

Problem Parameters:
  dt: 600
  t: 0
  t_final: 259200
  num_steps: 6
  num_windows: 72
  fric_law: mannings
  alpha: 0.0001454441043328608
  sol_var: h

Calculated Dimensions:
  State dimension: 1296
  Observation dimension: 23
  Total time steps: 433
  Observations per window: 6
  Number of observation obs_stations: 23

Matrix Information:
  Observation matrix H shape: (23, 1296)
  Background covariance B shape: (1296, 1296)
  Observation covariance R s

...

DCI_WME RMSE: 0.2290192802, DCI_WME, Relative Misfit: 0.0990
  ✓ Completed in 95.23s - Saved to dci_wme_analysis_obs_std_1.5_inflation_1.5.pkl

Experiment 24/24: obs_std_1.5_inflation_4.0
Selected 23 stations using method 'indices'
Station cells: [  1   3   5   7   9  11  13  15  17  19 103 105 107 109 111 113 115 117
 119 121 123 125 127]
DATA ASSIMILATION EXPERIMENT SETUP

Input Parameters:
  Pickle path: true_signal.pkl
  Observation std deviation: 1.5
  Observation time frequency: 1
  Final time: 259200
  Inflation factor: 4.0

Problem Parameters:
  dt: 600
  t: 0
  t_final: 259200
  num_steps: 6
  num_windows: 72
  fric_law: mannings
  alpha: 0.0001454441043328608
  sol_var: h

Calculated Dimensions:
  State dimension: 1296
  Observation dimension: 23
  Total time steps: 433
  Observations per window: 6
  Number of observation obs_stations: 23

Matrix Information:
  Observation matrix H shape: (23, 1296)
  Background covariance B shape: (1296, 1296)
  Observation covariance R shap

Processing windows:  64%|██████▍   | 46/72 [00:56<01:28,  3.41s/window]

Optimization failed:


Optimization completed:
  Success: False
  Status: 2
  Message: ABNORMAL: 
  Final cost: 2.027706e+01
  Iterations: 1
  Function evaluations: 44
  Gradient norm at solution: 3.780414e-05

------------------------------------------------------------



Processing windows: 100%|██████████| 72/72 [01:33<00:00,  1.30s/window]


DCI_WME RMSE: 0.2290105668, DCI_WME, Relative Misfit: 0.0990
  ✓ Completed in 94.12s - Saved to dci_wme_analysis_obs_std_1.5_inflation_4.0.pkl

Parameter cross validation completed!
Successful: 22/24
Convergence failures: 2

Convergence failure parameters:
  obs_std=0.01, inflation_factor=1.5
  obs_std=0.01, inflation_factor=4.0

Parameter sweep results summary:
----------------------------------------
✓ obs_std_0.01_inflation_0.1: SUCCESS
✓ obs_std_0.01_inflation_0.5: SUCCESS
✗ obs_std_0.01_inflation_1.5: CONVERGENCE FAILURE
✗ obs_std_0.01_inflation_4.0: CONVERGENCE FAILURE
✓ obs_std_0.05_inflation_0.1: SUCCESS
✓ obs_std_0.05_inflation_0.5: SUCCESS
✓ obs_std_0.05_inflation_1.5: SUCCESS
✓ obs_std_0.05_inflation_4.0: SUCCESS
✓ obs_std_0.1_inflation_0.1: SUCCESS
✓ obs_std_0.1_inflation_0.5: SUCCESS
✓ obs_std_0.1_inflation_1.5: SUCCESS
✓ obs_std_0.1_inflation_4.0: SUCCESS
✓ obs_std_0.5_inflation_0.1: SUCCESS
✓ obs_std_0.5_inflation_0.5: SUCCESS
✓ obs_std_0.5_inflation_1.5: SUCCESS
✓ obs_s

In [9]:
result = load_pickle('setup_result.pkl')
wme_results = analyze_error_statistics(CONFIG['obs_std_values'], CONFIG['inflation_factor_values'], result, 
                            analysis_type='dci_wme')

Error Statistics for DCI_WME Analysis
Obs Std    Inflation    RMSE            Misfit         
------------------------------------------------------------
0.010      0.100        0.225716        0.095065       
0.010      0.500        0.220278        0.082663       
0.010      1.500        ERROR/NOT FOUND N/A            
0.010      4.000        ERROR/NOT FOUND N/A            
0.050      0.100        0.228866        0.098912       
0.050      0.500        0.228202        0.098329       
0.050      1.500        0.226853        0.096605       
0.050      4.000        0.224393        0.093059       
0.100      0.100        0.228984        0.099012       
0.100      0.500        0.228827        0.098879       
0.100      1.500        0.228341        0.098511       
0.100      4.000        0.227540        0.097644       
0.500      0.100        0.229022        0.099043       
0.500      0.500        0.229014        0.099038       
0.500      1.500        0.228998        0.099024       
0.500

In [5]:
result = load_pickle('setup_result.pkl')
wme_results = analyze_error_statistics(CONFIG['obs_std_values'], CONFIG['inflation_factor_values'], result, 
                            analysis_type='bayes')

Error Statistics for BAYES Analysis
Obs Std    Inflation    RMSE            Misfit         
------------------------------------------------------------
0.010      0.100        0.230414        0.095964       
0.010      0.500        ERROR/NOT FOUND N/A            
0.010      1.500        ERROR/NOT FOUND N/A            
0.010      4.000        ERROR/NOT FOUND N/A            
0.050      0.100        0.228867        0.098857       
0.050      0.500        0.228469        0.098146       
0.050      1.500        0.229142        0.096926       
0.050      4.000        0.234853        0.095895       
0.100      0.100        0.228983        0.098997       
0.100      0.500        0.228834        0.098812       
0.100      1.500        0.228476        0.098152       
0.100      4.000        0.228627        0.097488       
0.500      0.100        0.229022        0.099042       
0.500      0.500        0.229018        0.099036       
0.500      1.500        0.229006        0.099019       
0.500  

In [ ]:
# Create DataFrame from dictionary directly, then reset index
df = pd.DataFrame.from_dict(wme_results, orient='index')
df.index.names = ['obs_std', 'inflation_factor']
df = df.reset_index()
df.index.name = "BAYES: Window = 1 Hour"
df.to_csv('da_output/bayes_one_hour_results.csv', index=True)